# Expanded distribution network

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/expanded_network.ipynb)

A company assembles electric bicycles and distributes them in Spain. After an expansion it serves 46 retailers from 12 local warehouses, and it considers 8 sites where it can contract capacity for a regional warehouse. This notebook solves three selection problems on that network. Each is written with binary variables and one constraint for each element that has to be served:

1. Retailers, local warehouses, candidate sites and distances.
2. Which local warehouse serves each retailer: a generalized assignment problem.
3. Which candidate sites act as regional warehouses: a set covering problem.
4. The weekly routes of local warehouse 2, in Tarragona: a set partitioning problem.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders and drop-down lists): after changing a value, run the cell again. Outside Colab the form is a line such as `truck = 90`: edit the value and run the cell.

| Symbol | Meaning | Term | Unit |
|---|---|---|---|
| $R$, $r$ | Retailers | retailer | — |
| $L$, $l$ | Local warehouses | — | — |
| $N$, $n$ | Candidate sites for a regional warehouse | — | — |
| $R_2$ | Retailers that the generalized assignment gives to local warehouse 2 | — | — |
| $J$, $j$ | Candidate routes of local warehouse 2 | — | — |
| $d_r$ | Expected orders of retailer $r$ in a design year after the expansion | — | bikes/year |
| $K^n_l$ | Bikes that local warehouse $l$ can receive and send on in a year | node capacity | bikes/year |
| $c^x_{lr}$ | Cost of carrying one bike from local warehouse $l$ to retailer $r$ | transport cost | €/bike |
| $a_{ln}$ | 1 if the road travel time from site $n$ to local warehouse $l$ is at most the maximum travel time, and 0 otherwise | — | — |
| $c^F_n$ | Cost of contracting capacity as a regional warehouse at site $n$ for a year | fixed cost | thousand €/year |
| $a_{rj}$ | 1 if route $j$ visits retailer $r$, and 0 otherwise | — | — |
| $c_j$ | Cost of route $j$ | — | € |
| $x_{lr}$ | 1 if local warehouse $l$ serves retailer $r$, and 0 otherwise | — | — |
| $y_n$ | 1 if site $n$ acts as a regional warehouse, and 0 otherwise | — | — |
| $y_j$ | 1 if route $j$ is driven in the week, and 0 otherwise | — | — |

The [document of this case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/expanded_network.md) explains all the data and results. The places, the populations, the distances and the travel times are real, except the urban distance of 10 km between a local warehouse and a retailer of the same city; the other data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import expanded_network as en

## 1. Retailers, local warehouses, candidate sites and distances

The first table gives each retailer, its city and its expected orders in a design year, $d_r$. The second gives each local warehouse and its node capacity for a year, $K^n_l$: the bikes it can receive and send on in a year. The third gives the transport cost of serving one bike of each retailer from each local warehouse, $c^x_{lr}$: the road distance times 0,010 €/(bike·km). A local warehouse is not at the same point as the shop it serves: for a retailer in the same city, the distance is taken as a fixed urban distance of 10 km. The fourth and the fifth give the candidate sites, with the fixed cost of contracting capacity at each one for a year and the road travel time from each site to each local warehouse. The last one gives the weekly orders of the six retailers of local warehouse 2.

In [ ]:
# @title Retailers, local warehouses, candidate sites and distances { display-mode: "form" }
en.show_data()

## 2. Which local warehouse serves each retailer

Each retailer is served, entire, from one local warehouse, and no local warehouse moves more bikes in a year than its node capacity:

$$
\begin{aligned}
\min \quad & \sum_{l \in L} \sum_{r \in R} c^x_{lr}\, d_r\, x_{lr} \\
\text{s.t.} \quad & \sum_{l \in L} x_{lr} = 1 && \forall r \in R && \text{(one local warehouse for each retailer)} \\
& \sum_{r \in R} d_r\, x_{lr} \le K^n_l && \forall l \in L && \text{(node capacity)} \\
& x_{lr} \in \{0, 1\} && \forall l \in L,\ r \in R
\end{aligned}
$$

The cell compares four plans: each retailer at its cheapest local warehouse, whatever the capacities; a rule that takes the retailers in the order of the list and sends each to its cheapest local warehouse with room left; the generalized assignment; and the transportation problem, the same model with $x_{lr}$ continuous between 0 and 1, the share of the orders of retailer $r$ that local warehouse $l$ serves. Choose a local warehouse and its node capacity. The first table lists the retailers that are not at their cheapest local warehouse in some plan, and the second the load of each local warehouse in each plan. On the map, a red line joins a retailer to a local warehouse that is not its cheapest one.

Raise the capacity of Girona to 2.700. Does the rule find a valid plan? Now set Girona back to 2.500 and lower Tarragona to 12.000. Does the rule still find a valid plan? Does the model?

In [ ]:
# @title Four plans for the node capacity of a local warehouse { display-mode: "form" }
warehouse = "Local warehouse 2"  # @param ["Local warehouse 1", "Local warehouse 2", "Local warehouse 3", "Local warehouse 4", "Local warehouse 5", "Local warehouse 6", "Local warehouse 7", "Local warehouse 8", "Local warehouse 9", "Local warehouse 10", "Local warehouse 11", "Local warehouse 12"]
capacity = 13000  # @param {type:"slider", min:1000, max:16000, step:100}
en.assignment(warehouse, capacity);

## 3. Which candidate sites act as regional warehouses

Each candidate site defines a candidate set: the local warehouses within the maximum road travel time of it, $a_{ln} = 1$. The model chooses the sites that act as regional warehouses, so that every local warehouse is within reach of at least one of them, at the least fixed cost:

$$
\begin{aligned}
\min \quad & \sum_{n \in N} c^F_n\, y_n \\
\text{s.t.} \quad & \sum_{n \in N} a_{ln}\, y_n \ge 1 && \forall l \in L && \text{(each local warehouse within reach)} \\
& y_n \in \{0, 1\} && \forall n \in N
\end{aligned}
$$

Choose the maximum travel time and the constraint on each local warehouse: «at least one» chosen site, the set covering problem, or «exactly one», a set partitioning problem on the same sets. The table gives the candidate set of each site; below, the sites chosen, their cost and the local warehouses within reach of two or more of them. On the map, the chosen sites are filled and joined to the local warehouses within their reach; a local warehouse in red is within reach of two or more chosen sites.

With 4 h, compare «at least one» and «exactly one». Then lower and raise the maximum travel time. From which time does every local warehouse have a site within reach, and how do the number of regional warehouses and their cost change?

In [ ]:
# @title Regional warehouses for a maximum travel time { display-mode: "form" }
max_time = 4  # @param {type:"slider", min:2, max:7, step:0.25}
constraint = "at least one"  # @param ["at least one", "exactly one"]
en.covering(max_time, constraint == "exactly one");

## 4. The weekly routes of local warehouse 2

Each week a small truck leaves local warehouse 2, in Tarragona, visits some of its six retailers and returns. A candidate route carries the weekly orders of its retailers within the capacity of the truck and drives at most 600 km; it visits them in the order of least distance, and costs 200 € plus 0,80 €/km. The candidate routes are listed with their cost before the model is solved, and the model chooses among them:

$$
\begin{aligned}
\min \quad & \sum_{j \in J} c_j\, y_j \\
\text{s.t.} \quad & \sum_{j \in J} a_{rj}\, y_j = 1 && \forall r \in R_2 && \text{(each retailer on exactly one route)} \\
& y_j \in \{0, 1\} && \forall j \in J
\end{aligned}
$$

Choose the capacity of the truck and the list of routes: the full list, or the short list of the routes of one or two retailers. The first table gives the candidate routes; below, the routes chosen with «exactly one», the set partitioning problem, and the plan with «at least one», the set covering problem on the same list. The map draws the routes chosen as straight lines between the places.

With a truck of 90 bikes, compare the full list and the short list: which route of the full list is missing from the short one, and what does its absence cost? Then lower the capacity of the truck. Below which capacity does the model have no solution, and why?

In [ ]:
# @title Weekly routes for a truck capacity { display-mode: "form" }
truck = 90  # @param {type:"slider", min:40, max:130, step:1}
routes = "full list"  # @param ["full list", "short list"]
en.routes(truck, routes == "full list");

The [document of this case](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/expanded_network.md) explains all the numbers and has a script that prints them.